# DCDP default inverse-problem simulations

This notebook runs DCDP pixel-space simulations with hyperparameter presets taken from the DCDP paper and official repository configs. It is set up to mirror your PDHG Colab data path, `/content/drive/MyDrive/mycode/test-ffhq`.

Sources used for the presets:
- DCDP paper arXiv:2403.06054v6, Appendix A/B: SGD momentum `0.9`, 20 DDIM purification steps, linear purification schedule, and Table 6 pixel-space defaults for linear inverse problems.
- Official DCDP repository YAMLs for runnable repo defaults and for tasks not tabulated in Appendix B, especially phase retrieval and random inpainting.

The paper defaults below are selected where available for DCDP solver hyperparameters. Measurement noise follows your inverse-problem suite: `sigma=0.05` for every task. The `inpainting_box` measurement itself uses your `mycode2` box definition, not the paper's 100x100 box. `inpainting_random` and `phase_retrieval` are repository/user-measurement presets, because Appendix B Table 6 reports box inpainting and does not tabulate phase retrieval. For quick Colab checks, the run cell defaults to Tweedie mode and skips metric sweeps/figures; switch `MODE` to `ddim` and enable metrics when you want the heavier paper-style DDIM run.

## Preset map

| Task | Measurement preset | Purification preset |
| --- | --- | --- |
| `super_resolution` | 4x downsampling, sigma `0.05` | paper Table 6 solver hyperparameters: `lr=1e3`, `K=10`, `tau=100`, `t1=400`, `tK=0` |
| `inpainting_box` | your `mycode2` box mask: `mask_len_range=[128,129]`, `margin=[32,32]`, sigma `0.05` | paper Table 6 solver hyperparameters: `lr=1e3`, `K=20`, `tau=50`, `t1=700`, `tK=0` |
| `gaussian_blur` | 61x61 Gaussian kernel, std/intensity `3.0`, sigma `0.05` | paper Table 6 solver hyperparameters: `lr=1e5`, `K=10`, `tau=100`, `t1=400`, `tK=0` |
| `motion_blur` | 61x61 motion blur, intensity `0.5`, sigma `0.05` | paper Table 6 solver hyperparameters: `lr=1e5`, `K=20`, `tau=50`, `t1=400`, `tK=0` |
| `inpainting_random` | repo random mask probability range `[0.3, 0.7]`, sigma `0.05` | repo inpainting purification defaults |
| `phase_retrieval` | repo oversample `2.0`, Gaussian measurement noise `0.05` | repo phase retrieval purification defaults: `lr=500`, `K=200`, `tau=100`, `t1=1000`, `tK=20` |

Note: the public repo's `motion_deblur` YAML uses `K=10`, `tau=100`. The notebook's `motion_blur` preset uses the paper Table 6 values by default.

In [ ]:
# @title 1. Clone the DCDP repo and install dependencies
# Use your fork/branch that contains this notebook and the scripts/configs added with it.
REPO_URL = "https://github.com/Seif-Hussein/Decoupled-Data-Consistency-with-Diffusion-Purification-for-Image-Restoration.git"  # @param {type:"string"}
BRANCH = "codex-dcdp-colab-defaults"  # @param {type:"string"}
REPO_DIR = "/content/dcdp"  # @param {type:"string"}
INSTALL_DEPS = True  # @param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(repo_dir)])
os.chdir(repo_dir)
print('Working directory:', Path.cwd())

if INSTALL_DEPS:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML', 'matplotlib', 'scipy', 'tqdm', 'scikit-image', 'torchmetrics[image]', 'lpips'])

# Motion deblurring in the original DCDP code imports this package directly.
if not Path('motionblur').exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/LeviBorodenko/motionblur', 'motionblur'])

required = [
    Path('scripts/run_dcdp_default_inverse_pipeline.py'),
    Path('task_configurations/dcdp_user_inpainting_box_config.yaml'),
    Path('purification_configurations/dcdp_paper_motion_deblur.yaml'),
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError('This checkout is missing the Colab preset files. Push/use the branch containing this notebook changes. Missing: ' + ', '.join(missing))

print('Setup ready.')

In [ ]:
# @title 2. Prepare checkpoint and Drive data
MOUNT_DRIVE = True  # @param {type:"boolean"}
FFHQ_CHECKPOINT_FROM_DRIVE = "/content/drive/MyDrive/ffhq_10m.pt"  # @param {type:"string"}
DOWNLOAD_CHECKPOINT_IF_MISSING = True  # @param {type:"boolean"}
DATASET_ROOT = "/content/drive/MyDrive/mycode/test-ffhq"  # @param {type:"string"}

from pathlib import Path
import shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

Path('models').mkdir(exist_ok=True)
target_ckpt = Path('models/ffhq_10m.pt')
drive_ckpt = Path(FFHQ_CHECKPOINT_FROM_DRIVE)
if not target_ckpt.exists() and drive_ckpt.exists():
    shutil.copy2(drive_ckpt, target_ckpt)
    print('Copied checkpoint to', target_ckpt)
elif target_ckpt.exists():
    print('Checkpoint found:', target_ckpt)
elif DOWNLOAD_CHECKPOINT_IF_MISSING:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    run(['gdown', '--id', '1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh', '-O', str(target_ckpt)])
    print('Downloaded checkpoint to', target_ckpt)
else:
    print('Checkpoint not found. Put ffhq_10m.pt at models/ffhq_10m.pt or set FFHQ_CHECKPOINT_FROM_DRIVE.')

data_root = Path(DATASET_ROOT)
if not data_root.exists():
    raise FileNotFoundError(f'DATASET_ROOT does not exist: {data_root}')
print('Images found:', len(list(data_root.rglob('*.png'))), 'in', data_root)

In [ ]:
# @title 3. Run one or more simulations
TASKS = "super_resolution"  # @param ["super_resolution", "inpainting_box", "inpainting_random", "gaussian_blur", "motion_blur", "phase_retrieval", "all"] {allow-input: true}
MAX_IMAGES = 1  # @param {type:"integer"}
DATA_START_IDX = 0  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}
GPU = 0  # @param {type:"integer"}
SAVE_DIR = "purification_results/dcdp_defaults_colab"  # @param {type:"string"}
MODE = "tweedie"  # @param ["tweedie", "ddim"]
SKIP_METRICS = True  # @param {type:"boolean"}
SAVE_MEASUREMENTS = False  # @param {type:"boolean"}
SAVE_PROGRESS_FIGURES = False  # @param {type:"boolean"}
DRY_RUN = False  # @param {type:"boolean"}

cmd = [
    sys.executable,
    'scripts/run_dcdp_default_inverse_pipeline.py',
    '--tasks', TASKS,
    '--gpu', str(GPU),
    '--seed', str(SEED),
    '--max-images', str(MAX_IMAGES),
    '--start-idx', str(DATA_START_IDX),
    '--dataset-root', DATASET_ROOT,
    '--save-dir', SAVE_DIR,
    '--mode', MODE,
]
if SKIP_METRICS:
    cmd.append('--skip-metrics')
if SAVE_MEASUREMENTS:
    cmd.append('--save-measurements')
if SAVE_PROGRESS_FIGURES:
    cmd.append('--save-progress-figures')
if DRY_RUN:
    cmd.append('--dry-run')
run(cmd)

In [ ]:
# Optional: list generated outputs
from pathlib import Path
root = Path(SAVE_DIR)
for path in sorted(root.rglob('*'))[:80]:
    print(path)